# FS-5 (ProtT5) - phase 1 on Kaggle

Runs the **FS-5 comparison profile only**: ProtT5 mean vectors, 1,024 columns,
on each of the four classifiers, with internal B-cell cross-validation. Four
runs.

This notebook is separate because FS-5 must not disturb what already ran.
Adding it to `FEATURE_SETS` would have changed the main sweep's planned run
count from 124 to 128 and phase 2's from 16 to 20, and both of those have
frozen, committed artifacts. Instead the profile is requested by name through
`comparison_feature_sets`, which switches the proposed and contrast arms off.

**FS-5 is a comparison profile.** It is reported and never selected from, so
nothing here writes or changes `frozen_configuration.json`. The
one-standard-error rule still sees only the proposed arm of the main sweep.

Its columns come from ProtT5 vectors, not from the CSV. **This notebook is
the one place the encoder ever runs** (cell 3a): it builds the vectors for
both B-cell and SARS, then checks that doing so left the pinned versions
intact. Phase 2 attaches what this session produced rather than building its
own, because float16 on GPU and float32 on CPU disagree by about 3e-4 and two
separate builds can land in two slightly different feature spaces.

SARS is embedded **label-free** -- only its peptide strings are read, never
its target column -- so phase 1 still leaves the SARS labels untouched, and
those vectors enter no phase 1 run.

`/ML/assets` is in `.gitignore`, so nothing arrives with the clone. Upload the
two `.npz` files this notebook writes and attach them to phase 2.

**This notebook does not wait for the main phase 1.** FS-5 is a comparison
profile: it takes no feature combination from the selection record, only the
protocol constants, and those are CFG's defaults rather than anything a sweep
produces. Run the two side by side in separate sessions. If a selection
record happens to be present, cell 3 cross-checks the protocol against it.

What this answers: whether a pretrained protein language model, given the same
folds, the same tuning budget and the same metric, beats the five-column
frozen combination. SARS labels are not read.

In [ ]:
# ---- 1. Get the pipeline ----------------------------------------------------
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/azhar416/B-Cell-Epitopes.git"
PROJECT_DIR = Path("/kaggle/working/B-Cell-Epitopes")

if not (PROJECT_DIR / "ML" / "train.py").is_file():
    if PROJECT_DIR.exists():
        raise FileNotFoundError(
            f"{PROJECT_DIR} exists but holds no ML pipeline. Remove it, or point "
            "PROJECT_DIR at the repository root."
        )
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)])

if not (PROJECT_DIR / "ML" / "train.py").is_file():
    raise FileNotFoundError(f"Clone finished but {PROJECT_DIR}/ML/train.py is absent.")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


def _git(*arguments):
    return subprocess.run(
        ["git", "-C", str(PROJECT_DIR), *arguments],
        capture_output=True, text=True,
    ).stdout.strip()


# A clone left behind by an earlier step would be reused silently and run old
# code, so confirm it is the current master rather than assuming it.
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "fetch", "-q", "--depth", "1", "origin", "master"],
    capture_output=True, text=True,
)
head = _git("rev-parse", "HEAD")
remote = _git("rev-parse", "FETCH_HEAD") or _git("rev-parse", "origin/master")

print(f"Pipeline at {PROJECT_DIR}")
print(f"  commit in use : {head[:12]}")
print(f"  origin/master : {remote[:12] or 'could not be read'}")
if remote and head != remote:
    raise RuntimeError(
        "This clone is not at origin/master, so it would run outdated code. "
        f"In use {head[:12]}, remote {remote[:12]}. Delete {PROJECT_DIR} and "
        "re-run this cell."
    )
print("  -> up to date")
COMMIT = head

In [ ]:
# ---- 2. Dependencies and imports -------------------------------------------
import importlib.util

required = ("numpy", "pandas", "sklearn", "imblearn", "joblib", "matplotlib", "xgboost")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print(f"Installing: {missing}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_DIR / "ML" / "requirements.txt")]
    )

import numpy as np
import pandas as pd
from IPython.display import display, FileLink

from ML.CFG import CFG
from ML.collect_artifacts import collect_all
from ML.dataset import (
    PROTTRANS_DIMENSION,
    PROTTRANS_MODEL_NAME,
    PROTTRANS_TOKENIZER_NAME,
    load_labeled_dataset,
)
from ML.experiment_matrix import (
    MODEL_TYPES,
    build_experiment_matrix,
    report_environment,
    run_experiment_matrix,
)
from ML.summary_table import (
    build_internal_table,
    internal_header,
    read_summaries,
    render_internal_pretty,
    render_rows_tsv,
)

# scikit-learn decides the fold split and every estimator; pandas decides the
# row order feeding into it. FS-5's rows are printed in the same table as the
# tabular profiles', so a version change here would silently mix two
# protocols. The settings cell turns a mismatch into a hard stop.
ENVIRONMENT_OK = report_environment()
if not ENVIRONMENT_OK:
    print()
    print("Kaggle has refreshed its image since the other sweeps ran. Pin the")
    print("versions with the line below, restart the kernel, and run again:")
    print(f"    !pip install -q -r {PROJECT_DIR}/ML/requirements.txt")

## Settings

In [ ]:
# ---- 3. Settings -----------------------------------------------------------
import json
from collections import Counter

SWEEP_MODELS = MODEL_TYPES
COLLECT_PREDICTIONS = True

WORKING = Path("/kaggle/working")
RUN_DIR = WORKING / "ML-artifacts" / "fs5_phase1"
COLLECTED_DIR = WORKING / "collected" / "fs5_phase1"
RUN_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = RUN_DIR / "experiment_summary.csv"

# FS-5 is a COMPARISON profile: it is reported, never selected from, and it
# takes no feature combination from the selection record. All it shares with
# the other sweeps is the protocol -- the tuning metric and the seed -- and
# those are CFG's defaults, decided before any sweep ran rather than produced
# by one. So this notebook does NOT wait for phase 1: the two can run side by
# side in separate sessions.
TUNING_METRIC = CFG().HYPERPARAMETER_TUNING_METRIC

# If a selection record is already present, cross-check against it. A
# disagreement would mean the protocols really had diverged, which is worth
# stopping for; an absent record is not, because FS-5 needs nothing from it.
FROZEN_PATH = PROJECT_DIR / "ML" / "frozen_configuration.json"
frozen = None
if FROZEN_PATH.is_file():
    frozen = json.loads(FROZEN_PATH.read_text(encoding="utf-8"))
    divergence = []
    if frozen.get("phase1_tuning_metric") != TUNING_METRIC:
        divergence.append(
            f"tuning metric {frozen.get('phase1_tuning_metric')!r} there "
            f"against {TUNING_METRIC!r} here"
        )
    # The record is written from a CSV, so its protocol fields are strings.
    if int(frozen.get("phase1_random_state", 42)) != CFG().RANDOM_STATE:
        divergence.append(
            f"seed {frozen.get('phase1_random_state')} there against "
            f"{CFG().RANDOM_STATE} here"
        )
    if divergence:
        raise RuntimeError(
            "FS-5 would not be comparable with the other sweeps: "
            + "; ".join(divergence)
        )
    print(f"cross-checked against {FROZEN_PATH.name}: protocol matches")
else:
    print("No selection record present yet. FS-5 needs none, so this sweep "
          "can run alongside phase 1.")

if not ENVIRONMENT_OK:
    raise RuntimeError(
        "This image is not the one the other sweeps ran in, so FS-5's numbers "
        "could not be placed beside them. Pin the versions with "
        f"'!pip install -q -r {PROJECT_DIR}/ML/requirements.txt', restart the "
        "kernel, and run this notebook again."
    )


def _find_data_dir(filename):
    bundled = PROJECT_DIR / "dataset"
    if (bundled / filename).is_file():
        return bundled
    found = sorted(Path("/kaggle/input").rglob(filename))
    if len(found) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename}; found {len(found)}."
        )
    return found[0].parent


DATA_DIR = _find_data_dir("input_bcell.csv")

cfg = CFG(
    DATASET_NAME="bcell",
    TRAINING_MODE="same_dataset",
    FEATURE_SET="perbandingan_5",
    DATA_DIR=DATA_DIR,
    OUTPUT_DIR=RUN_DIR,
    HYPERPARAMETER_TUNING_METRIC=TUNING_METRIC,
)
cfg.validate()

# comparison_feature_sets names the profile and switches the proposed and
# contrast arms off, so this sweep cannot disturb the frozen artifacts of the
# sweeps that already ran. Adding FS-5 to FEATURE_SETS instead would have
# changed THEIR planned run counts, which is why it is not there.
planned = build_experiment_matrix(
    SWEEP_MODELS, comparison_feature_sets=("perbandingan_5",)
)
arms = Counter(spec.feature_set for spec in planned)

print(f"models: {list(SWEEP_MODELS)}")
print(f"{len(planned)} runs = {len(planned) // len(SWEEP_MODELS)} configuration x "
      f"{len(SWEEP_MODELS)} models")
for name, count in sorted(arms.items()):
    print(f"  {name:<16}{count // len(SWEEP_MODELS):3d} per model")
print()
print(f"mode={cfg.TRAINING_MODE} (internal B-cell CV only; SARS labels are not read)")
print(f"feature_set={cfg.FEATURE_SET} (ProtT5 mean vectors, "
      f"{PROTTRANS_DIMENSION} columns)")
print(f"tuning={cfg.HYPERPARAMETER_TUNING} metric={cfg.HYPERPARAMETER_TUNING_METRIC} "
      f"candidates/model={cfg.TUNING_CANDIDATES_PER_MODEL}")
print(f"class_weight={cfg.CLASS_WEIGHT} scale_pos_weight={cfg.XGBOOST_SCALE_POS_WEIGHT} "
      f"smote={cfg.USE_SMOTE} seed={cfg.RANDOM_STATE}")
print(f"data={DATA_DIR}")
print(f"out={RUN_DIR}")

expected_runs = 1 * len(SWEEP_MODELS)
if len(planned) != expected_runs:
    raise RuntimeError(
        f"Expected {expected_runs} runs (FS-5 on each model) but the matrix "
        f"planned {len(planned)}. Do not start the sweep until the plan is "
        "what you meant to run."
    )
if set(arms) != {"perbandingan_5"}:
    raise RuntimeError(f"The plan reaches outside FS-5: {sorted(arms)}.")
if cfg.RANDOM_STATE != 42:
    raise RuntimeError(
        f"Seed is {cfg.RANDOM_STATE}; every other sweep used 42, and the fold "
        "assignment depends on it."
    )
print()
print("Plan matches: FS-5 only, 4 models, nothing else touched.")
print()
print("COST: FS-5 carries 1,024 columns against 5 for the frozen combination,")
print(f"and each model is tuned over {cfg.TUNING_CANDIDATES_PER_MODEL} candidates "
      "before its 5 folds.")
print("Expect this sweep to take far longer per run than the tabular profiles did.")

## Build the ProtT5 vectors

The only cell in either FS-5 notebook that loads an encoder. It writes both
caches, then re-checks that doing so did not disturb the pinned versions.
Attach the two `.npz` files it produces to phase 2 as a Kaggle dataset.

In [ ]:
# ---- 3a. Build the ProtT5 vectors ------------------------------------------
# This is the ONE place the encoder ever runs. Phase 2 does not build; it
# attaches what this session produced. That is deliberate: float16 on GPU and
# float32 on CPU disagree by about 3e-4, so vectors built in two different
# sessions can land in two slightly different feature spaces, and the external
# comparison would be made across them without any visible symptom.
#
# BOTH datasets are built here for the same reason. SARS is embedded
# label-free -- only its peptide strings are read, never its target column --
# so phase 1 still leaves the SARS labels untouched. The vectors are a fixed
# function of the peptide and enter no phase 1 run.
import subprocess as _subprocess

BUILD_DEVICE = "auto"      # "cpu" for float32, "cuda" for float16 and speed
REBUILD = False            # True to overwrite caches that are already there

command = [
    sys.executable, "-m", "ML.build_prott5_assets",
    "--datasets", "bcell", "sars",
    "--label-free", "sars",
    "--data-dir", str(DATA_DIR),
    "--device", BUILD_DEVICE,
    "--batch-size", "32",
]
if REBUILD:
    command.append("--force")

print(" ".join(command[2:]))
print()
completed = _subprocess.run(command, cwd=str(PROJECT_DIR), text=True)
if completed.returncode != 0:
    raise RuntimeError(
        "Building the ProtT5 vectors failed. If the encoder could not be "
        "imported, install it with 'pip install torch transformers "
        "sentencepiece' and run this cell again."
    )

# Importing torch can pull a different numpy or pandas in behind it, and the
# fold split depends on those. Re-check rather than assume the build left the
# pinned image intact.
if not report_environment():
    raise RuntimeError(
        "Building the vectors changed the installed versions, so the sweep "
        "would no longer match the other phases. Restore them with "
        f"'!pip install -q -r {PROJECT_DIR}/ML/requirements.txt', restart the "
        "kernel, and run again -- the caches are already written, so the "
        "build cell will skip."
    )
print()
print("Environment still matches; the vectors are ready.")

## The ProtT5 cache, before anything is trained

In [ ]:
# ---- 3b. ProtT5 asset preflight --------------------------------------------
# FS-5's columns are not computed from the CSV: they are read from ProtT5
# vectors under ML/assets/perbandingan_5, which the clone now carries. A
# missing or mismatched cache is the one failure that would otherwise surface
# hours into the sweep, so it is settled here.
ASSET_DIR = PROJECT_DIR / "ML" / "assets" / "perbandingan_5"
ARRAY_NAMES = ("sequences.npy", "embeddings.npy", "model_name.npy",
               "tokenizer_name.npy")
NEEDED = ("bcell",) + ()


# The cache exists in three shapes on disk and the loader reads all three, so
# the preflight accepts all three too: a single .npz, the same four arrays
# unpacked into a <dataset>_prott5/ folder, or those arrays loose in the asset
# directory. Which one is present is a packaging decision -- a cache too large
# to commit as one blob ends up unpacked -- not a difference in the vectors.
def _shape_on_disk(dataset_name):
    if (ASSET_DIR / f"{dataset_name}_prott5.npz").is_file():
        return "npz", ASSET_DIR / f"{dataset_name}_prott5.npz"
    folder = ASSET_DIR / f"{dataset_name}_prott5"
    if all((folder / name).is_file() for name in ARRAY_NAMES):
        return "folder", folder
    if dataset_name == "bcell" and all(
        (ASSET_DIR / name).is_file() for name in ARRAY_NAMES
    ):
        return "folder", ASSET_DIR
    return None, None


def _read_header(kind, path):
    if kind == "npz":
        with np.load(path, allow_pickle=False) as cache:
            return (cache["embeddings"].shape,
                    str(cache["model_name"].item()),
                    str(cache["tokenizer_name"].item()))
    return (
        np.load(path / "embeddings.npy", allow_pickle=False, mmap_mode="r").shape,
        str(np.load(path / "model_name.npy", allow_pickle=False).item()),
        str(np.load(path / "tokenizer_name.npy", allow_pickle=False).item()),
    )


print(f"asset directory: {ASSET_DIR}")
headers = {}
for dataset_name in NEEDED:
    kind, path = _shape_on_disk(dataset_name)
    if kind is None:
        raise FileNotFoundError(
            f"No ProtT5 cache for {dataset_name} under {ASSET_DIR}. Expected "
            f"{dataset_name}_prott5.npz, or a {dataset_name}_prott5/ folder "
            f"holding {list(ARRAY_NAMES)}. Build it with "
            "'python -m ML.build_prott5_assets'."
        )
    shape, model_name, tokenizer_name = _read_header(kind, path)
    if model_name != PROTTRANS_MODEL_NAME:
        raise RuntimeError(
            f"{dataset_name} encoder is {model_name!r}, expected "
            f"{PROTTRANS_MODEL_NAME!r}."
        )
    if tokenizer_name != PROTTRANS_TOKENIZER_NAME:
        raise RuntimeError(
            f"{dataset_name} tokenizer is {tokenizer_name!r}, expected "
            f"{PROTTRANS_TOKENIZER_NAME!r}."
        )
    if shape[1] != PROTTRANS_DIMENSION:
        raise RuntimeError(
            f"{dataset_name} vectors are {shape[1]}-dimensional, expected "
            f"{PROTTRANS_DIMENSION}."
        )
    headers[dataset_name] = shape
    print(f"  {dataset_name:<6} {shape[0]:,} x {shape[1]} vectors  "
          f"({path.name if kind == 'npz' else path.name + '/'})")
print(f"  encoder   {PROTTRANS_MODEL_NAME}")
print(f"  tokenizer {PROTTRANS_TOKENIZER_NAME}")

# Provenance cross-check, for the sidecar JSON that build_prott5_assets
# writes. This used to be done by comparing the vectors of peptides present in
# BOTH datasets, but cleaning now removes that overlap, so there is nothing
# left to compare and that check became dead code. What the sidecars carry is
# what actually matters: float16 on GPU and float32 on CPU disagree by about
# 3e-4, so caches built on different devices sit in different feature spaces,
# and the external comparison would be made across them with no symptom.
provenance = {}
for dataset_name in NEEDED:
    sidecar = ASSET_DIR / f"{dataset_name}_prott5.json"
    if sidecar.is_file():
        provenance[dataset_name] = json.loads(sidecar.read_text(encoding="utf-8"))

if len(provenance) == len(NEEDED) and len(NEEDED) > 1:
    first, *rest = NEEDED
    disagreement = [
        f"{field}: {provenance[first].get(field)!r} against "
        f"{provenance[other].get(field)!r}"
        for other in rest
        for field in ("encoder", "tokenizer", "pooling", "device", "dtype")
        if provenance[first].get(field) != provenance[other].get(field)
    ]
    if disagreement:
        raise RuntimeError(
            "The caches were not built the same way, so the datasets are not "
            "in one feature space: " + "; ".join(disagreement)
            + ". Rebuild them in one run of ML.build_prott5_assets."
        )
    device = provenance[first].get("device")
    dtype = provenance[first].get("dtype")
    if device in (None, "auto") or dtype is None:
        # Older sidecars recorded the REQUEST, so "auto" says nothing about
        # what it resolved to and two different devices would both match.
        print("  provenance  encoder, tokenizer and pooling agree, but the "
              "sidecars record")
        print(f"              device={device!r} dtype={dtype!r}, so a GPU/CPU "
              "mix cannot be ruled")
        print("              out from them. Safe if both came from one "
              "build command.")
    else:
        print(f"  provenance  all caches built on {device!r} in {dtype}, same "
              "encoder and pooling")
elif len(NEEDED) == 1 and provenance:
    # One cache, so there is nothing to cross-check; just say what it is.
    only = provenance[NEEDED[0]]
    print(f"  provenance  built on {only.get('device')!r} in "
          f"{only.get('dtype', 'an unrecorded dtype')}, pooling "
          f"{only.get('pooling')!r}")
elif provenance:
    print("  provenance  sidecar JSON present for only some caches, so a "
          "device mix cannot be ruled out.")
else:
    print("  provenance  no sidecar JSON, so how these were built is not "
          "recorded.")

for dataset_name, shape in headers.items():
    if dataset_name in provenance:
        recorded = provenance[dataset_name].get("number_of_sequences")
        if recorded is not None and recorded != shape[0]:
            raise RuntimeError(
                f"{dataset_name}: the cache holds {shape[0]:,} vectors but its "
                f"sidecar records {recorded:,}, so they do not describe the "
                "same build."
            )
print()
print("ProtT5 cache present and matches the encoder the loader demands.")

## Restarting a run that ran out of time

In [ ]:
# ---- 4. Restart an interrupted session (normally skip) ---------------------
import shutil

# Example, when continuing a session that timed out:
#   PREVIOUS_OUTPUT = "/kaggle/input/bcell-phase1/ML-artifacts/phase1"
#   RESUME = True
PREVIOUS_OUTPUT = None
RESUME = False

if PREVIOUS_OUTPUT is not None:
    source = Path(PREVIOUS_OUTPUT)
    if not source.is_dir():
        raise FileNotFoundError(f"PREVIOUS_OUTPUT is not a directory: {source}")
    copied = 0
    for item in source.rglob("*"):
        if not item.is_file():
            continue
        target = RUN_DIR / item.relative_to(source)
        if target.exists():
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, target)
        copied += 1
    print(f"Restored {copied} file(s) into {RUN_DIR}")
    if not RESUME:
        print("WARNING: RESUME is False, so the restored runs will be repeated.")
elif RESUME:
    if not list(RUN_DIR.rglob("*_metrics.json")):
        print("WARNING: RESUME is True but RUN_DIR holds no reports, so nothing "
              "will be skipped. Set PREVIOUS_OUTPUT to the attached run directory.")
else:
    print("Fresh run; nothing to restore.")

## Check the data before spending hours on it

In [ ]:
# ---- 5. Data sanity check --------------------------------------------------
# Loading with feature_set="perbandingan_5" is itself the strict check: the
# loader refuses a cache whose encoder differs, and refuses to proceed if any
# cleaned peptide has no vector.
bcell = load_labeled_dataset(
    "bcell", cfg.DATA_DIR,
    max_peptide_length=cfg.MAX_PEPTIDE_LENGTH, feature_set="perbandingan_5",
)
counts = bcell.target.value_counts().sort_index().to_dict()
proteins = int(bcell.parent_protein_id.nunique())
vector_columns = [name for name in bcell.columns if name.startswith("prott5_")]
print(f"B-cell: {len(bcell):,} rows, {proteins:,} proteins, classes {counts}")
print(f"        {len(vector_columns)} ProtT5 columns")

expected = (14323, 758, {0: 10453, 1: 3870})
actual = (len(bcell), proteins, counts)
if actual != expected:
    print(f"WARNING: expected {expected} but got {actual}. The input file differs "
          "from the one earlier results came from.")
else:
    print("Matches the reference cleaning counts.")
if len(vector_columns) != PROTTRANS_DIMENSION:
    raise RuntimeError(f"Expected {PROTTRANS_DIMENSION} ProtT5 columns.")
if not np.isfinite(bcell[vector_columns].to_numpy(dtype=float)).all():
    raise RuntimeError("ProtT5 columns contain non-finite values.")

print("SARS is not read in phase 1, so it stays an untouched external test set.")

## Run

In [ ]:
# ---- 6. Run ----------------------------------------------------------------
import time

started = time.time()
matrix_rows = run_experiment_matrix(
    RUN_DIR,
    base_config=cfg,
    model_types=SWEEP_MODELS,
    comparison_feature_sets=("perbandingan_5",),
    tuning_metric=TUNING_METRIC,
    resume=RESUME,
)
elapsed = time.time() - started

matrix_summary = pd.DataFrame(matrix_rows)
ok = int((matrix_summary["status"] == "success").sum())
print()
print(f"{len(matrix_summary)} runs in {elapsed / 3600:.2f} h -- "
      f"{ok} successful, {len(matrix_summary) - ok} failed")

failures = matrix_summary[matrix_summary["status"] != "success"]
if not failures.empty:
    display(failures[["experiment_id", "error"]])
display(FileLink(str(SUMMARY_PATH)))

## Collect every measurement

In [ ]:
# ---- 7. Collect metrics ----------------------------------------------------
manifest = collect_all(
    [SUMMARY_PATH], COLLECTED_DIR, include_predictions=COLLECT_PREDICTIONS
)

print(f"Collected {manifest['runs_collected']} run(s) into {COLLECTED_DIR}\n")
for name, info in manifest["tables"].items():
    size_mb = (COLLECTED_DIR / info["file"]).stat().st_size / 1048576
    print(f"  {name:<28} {info['rows']:>9,} rows  {size_mb:>7.1f} MB  {info['file']}")
    display(FileLink(str(COLLECTED_DIR / info["file"])))

if manifest["runs_missing_report"]:
    print(f"\nNo readable report for {len(manifest['runs_missing_report'])} run(s):")
    for experiment_id in manifest["runs_missing_report"]:
        print(f"  {experiment_id}")

display(FileLink(str(COLLECTED_DIR / "manifest.json")))

## The table

In [ ]:
# ---- 8. Internal comparison table -----------------------------------------
internal_rows = build_internal_table(
    read_summaries([SUMMARY_PATH]), model_order=SWEEP_MODELS
)
print(render_internal_pretty(internal_rows))

TSV_PATH = COLLECTED_DIR / "internal_table_fs5.tsv"
TSV_PATH.write_text(render_rows_tsv(internal_header(), internal_rows),
                    encoding="utf-8")
filled = sum(1 for row in internal_rows if row[2])
print(f"\n{filled}/{len(internal_rows)} rows have internal metrics.")
print("Download this and paste it into the spreadsheet:")
display(FileLink(str(TSV_PATH)))

## Drop the model bundles

In [ ]:
# ---- 9. Reclaim space ------------------------------------------------------
DROP_MODEL_BUNDLES = True
DROP_PLOTS = False          # per-run PNGs; keep them if you want the curves

def _purge(pattern):
    files = list(RUN_DIR.rglob(pattern))
    freed = sum(path.stat().st_size for path in files)
    for path in files:
        path.unlink()
    return len(files), freed / 1048576

def _tree_size(root):
    return sum(p.stat().st_size for p in root.rglob("*") if p.is_file()) / 1048576

print(f"Before: {_tree_size(RUN_DIR):,.1f} MB in {RUN_DIR}")
if DROP_MODEL_BUNDLES:
    count, freed = _purge("*.joblib")
    print(f"  removed {count} model bundles ({freed:,.1f} MB)")
if DROP_PLOTS:
    count, freed = _purge("*.png")
    print(f"  removed {count} plots ({freed:,.1f} MB)")
print(f"After:  {_tree_size(RUN_DIR):,.1f} MB")
print(f"Collected tables: {_tree_size(COLLECTED_DIR):,.1f} MB in {COLLECTED_DIR}")